In [ ]:
import geopandas as gpd
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
from rasterio.warp import Resampling
import re
from rioxarray.exceptions import NoDataInBounds, OneDimensionalRaster
from shapely.geometry import box
from cryoswath_paper_workflow.workflow_support import snakemake_fallback_builder, strip_spec_chars, project_path
from cryoswath.misc import (
    rgi_code_translator,
    dem_path,
    l3_path,
)
import xarray as xr

snakemake_fallback = snakemake_fallback_builder(globals())

In [ ]:
def strip_filter(arr, axis=None):
    abs_max = np.nanmax(np.abs(arr), axis=axis)
    spread = np.nanmax(arr, axis=axis) - np.nanmin(arr, axis=axis)
    return (abs_max < 150) & (spread < 150)

In [ ]:
if "snakemake" in globals():
    o1 = int(str(globals()["snakemake"].wildcards.token)[:2])
else:
    raw = input("o1 region code (e.g. 1): ").strip()
    o1 = int(raw or "1") 

In [ ]:
refdata_lockfile = Path(
    snakemake_fallback(
        "input",
        0,
        Path(
            "resources",
            "calval",
            "select_arcticdem_strips__"
            + rgi_code_translator(f"{o1:02d}", out_type="long_code"),
        ).with_suffix(".lock.feather"),
    )
)
refdata_ids = gpd.read_feather(refdata_lockfile)

In [ ]:
def drop_timedups_select_more_vals(da):
    if len(da.time) == 1:
        return da
    tix = da.count(["x", "y"]).argmax("time").values
    return da.isel(time=[tix])

In [ ]:
strip_corr_collection = []
med_iqr_cnt_df = pd.DataFrame(
    index=pd.MultiIndex.from_product(
        [["dummy"], [0], range(7)],
        names=["strip", "intersec_idx", "flag"],
    ),
    columns=["_median", "_iqr", "_count"],
)
for point_id, row in refdata_ids.iterrows():
    item_list = row.strip_ids
    da_collection = []
    for item_id in item_list:
        strip = xr.open_dataarray(
            Path(dem_path, "strips") / (item_id + "__32m_mean.tif"), chunks={}
        )
        month = pd.to_datetime([item_id.split("_")[3]], format="%Y%m%d").snap(
            "MS"
        )
        idx_cnt = 0
        for file in Path(l3_path).glob(
            "__".join(
                [
                    "Glacier_surface_elevation",
                    strip_spec_chars(rgi_code_translator(f"{o1:02d}")),
                    "*",
                    "monthly_500x500m.nc",
                ]
            )
        ):
            if len(file.name.split("__")) != 4:  # all concerned files conform <type>__<o1>__<o2>__<units>.nc
                continue                         # skipping closeups (e.g. Vatnajoekull)
            ds = xr.open_dataset(file, chunks={}).sel(time=month)
            if box(*strip.rio.bounds()).intersects(
                box(*ds.rio.transform_bounds(strip.rio.crs))
            ):
                try:
                    clip = ds.rio.clip_box(
                        *box(*strip.rio.transform_bounds(ds.rio.crs)).bounds
                    )
                except (NoDataInBounds, OneDimensionalRaster):
                    continue
                strip_repr = (
                    strip.rio.reproject_match(clip, resampling=Resampling.average)
                    - clip.elev_diff_ref
                )
                if (strip_repr.pipe(np.abs) > 150).any():
                    _, axes = plt.subplots(1, 2)
                    strip_repr.plot(ax=axes[0], robust=True)
                    plt.title("")
                    mask = strip_repr.rolling(
                        x=5, y=5, center=True, min_periods=1
                    ).reduce(strip_filter)
                    mask.plot(ax=axes[1])
                    plt.title("")
                    plt.suptitle(item_id)
                    plt.show()
                    strip_repr = strip_repr.where(mask)
                clip["error"] = clip.elev_diff - strip_repr
                if clip.error.count() < 6:
                    continue
                clip = clip.where(clip.error.notnull()).compute()
                da_collection.append(clip)
                for label, grp in clip.groupby("elev_diff_interp_flag"):
                    if grp.error.count() < 6:
                        continue
                    grp = grp.unstack().reindex_like(clip)
                    coarsened_err = (
                        grp.error.coarsen(x=4, y=4, boundary="pad")
                        .mean()
                        .squeeze()
                        .compute()
                    )
                    med = coarsened_err.median().item()
                    iqr = (
                        coarsened_err.quantile([0.25, 0.75]).diff("quantile").item()
                    )
                    cnt = coarsened_err.count().item()
                    med_iqr_cnt_df.loc(0)[item_id, idx_cnt, int(label)] = (
                        med,
                        iqr,
                        cnt,
                    )
    if not da_collection:
        continue
    da = xr.concat([da.error.isel(band=0).drop_vars("band") for da in da_collection], "time", join="outer").sortby("time")
    da = da.groupby("time").map(drop_timedups_select_more_vals)
    strip_error_correlation = pd.Series(index=pd.MultiIndex.from_product([da.time.values, da.time.values]))
    for t1, t2 in combinations(da.time.values, 2):
        tmp = da.sel(time=[t1, t2]).stack(xy=["x", "y"]).dropna("xy", how="any")
        strip_error_correlation.loc[t1, t2] = xr.corr(tmp.isel(time=0), tmp.isel(time=1)).values
    strip_error_correlation.to_xarray().plot()
    plt.title("Error correlation through time\n(around point"f"{point_id})")
    plt.show()
    strip_corr_collection.append(strip_error_correlation.dropna())
strip_error_correlation = pd.concat(strip_corr_collection, keys=refdata_ids.index, names=["id", "t1", "t2"])
med_iqr_cnt_df.drop("dummy", level="strip", inplace=True)

In [ ]:
print(med_iqr_cnt_df)

In [ ]:
med_iqr_cnt_df.to_csv(
    snakemake_fallback(
        "output",
        0,
        str(project_path("resources", f"calval__compare_arcticdem_strips__interpolation_type__{rgi_code_translator(f'{o1:02d}', out_type='long_code')}.csv")),
    )
)

In [ ]:
strip_error_correlation.to_csv(
    snakemake_fallback(
        "output",
        1,
        str(project_path("resources", f"calval__compare_arcticdem_strips__temporal_correlation__{rgi_code_translator(f'{o1:02d}', out_type='long_code')}.csv")),
    )
)